In [17]:
# Day 20 — End-to-end pipeline test, 10 fraud + 10 legit
import time
import numpy as np
import pandas as pd
from collections import Counter
from pipeline import predict_and_explain

# Raw test split — pipeline.py does feature engineering internally now,
# so this must be the RAW columns (pre-FE), not the engineered X_test
df_test = pd.read_csv('../data/processed/test.csv')   # confirm this path matches your layout
y_test  = df_test['isFraud']

fraud_txns = df_test[y_test == 1].head(10).to_dict('records')
legit_txns = df_test[y_test == 0].head(10).to_dict('records')
test_txns  = fraud_txns + legit_txns

print("=== WITH NARRATIVE (Groq LLM call included) ===\n")
latencies_with = []
tier_counts    = Counter()   # not hardcoded — you have 4 tiers now (LOW/MODERATE/ELEVATED/HIGH), not 3
correct        = 0

for txn in test_txns:
    actual = int(txn['isFraud'])
    result = predict_and_explain(txn, generate_narrative=True)
    latencies_with.append(result['pipeline_latency_ms'])

    tier       = result['risk_tier']['tier']
    is_fraud_p = result['is_fraud']
    tier_counts[tier] += 1
    correct += int(is_fraud_p == bool(actual))

    print(f"TxnID: {str(result['transaction_id']):<12} | "
          f"Score: {result['fraud_score']:.4f} | "
          f"Tier: {tier:<14} | "
          f"Actual: {'FRAUD' if actual else 'LEGIT':<6} | "
          f"Latency: {result['pipeline_latency_ms']:.0f}ms")

    if result['narrative']:
        brief = result['narrative']['investigation_brief']
        has_notice = result['narrative']['compliance_notice'] is not None
        print(f"  Brief   : {brief[:110]}...")
        print(f"  Escalation notice injected: {has_notice}")

    if tier == 'HIGH RISK' and result['narrative']:
        notice = result['narrative']['compliance_notice'] or ''
        assert 'POCA 2002' in notice, f"HIGH RISK case {result['transaction_id']} missing POCA 2002 reference"
        assert 'as soon as is practicable' in notice, f"HIGH RISK case {result['transaction_id']} missing s.330 phrase"
        assert 'does not confirm' in notice, "HIGH RISK case missing disclaimer language"
        print("  ✅ POCA 2002 s.330 language verified")

    # Your own design rule: is_fraud=True should never land LOW RISK
    if actual == 1 and tier == 'LOW RISK':
        print("  ⚠️  confirmed fraud landed in LOW RISK — investigate")
    print()

print(f"\n--- SUMMARY ---")
print(f"Sample accuracy (20 cases)   : {correct/len(test_txns):.0%}")
print(f"Avg latency (with narrative) : {np.mean(latencies_with):.0f}ms")
print(f"Max latency (with narrative) : {np.max(latencies_with):.0f}ms")
print(f"Tier distribution            : {dict(tier_counts)}")

print("\n\n=== WITHOUT NARRATIVE (ML pipeline only) ===\n")
latencies_without = [
    predict_and_explain(txn, generate_narrative=False)['pipeline_latency_ms']
    for txn in test_txns
]
print(f"Avg latency (ML only) : {np.mean(latencies_without):.0f}ms")
print(f"Max latency (ML only) : {np.max(latencies_without):.0f}ms")
print(f"LLM overhead (avg)    : {np.mean(latencies_with) - np.mean(latencies_without):.0f}ms")

=== WITH NARRATIVE (Groq LLM call included) ===



c:\Users\sathi\fraud-platform\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


TxnID: 3518517      | Score: 0.0650 | Tier: LOW RISK       | Actual: FRAUD  | Latency: 8994ms
  Brief   : This transaction has been assessed as LOW RISK with a fraud risk score of 0.065, indicating that no immediate ...
  Escalation notice injected: False
  ⚠️  confirmed fraud landed in LOW RISK — investigate



c:\Users\sathi\fraud-platform\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


TxnID: 3518582      | Score: 0.3900 | Tier: MODERATE RISK  | Actual: FRAUD  | Latency: 13671ms
  Brief   : This transaction has been classified as MODERATE RISK with a fraud risk score of 0.390, indicating a need for ...
  Escalation notice injected: False



...

TxnID: 3518494      | Score: 0.0100 | Tier: LOW RISK       | Actual: LEGIT  | Latency: 14441ms
  Brief   : This transaction has been assessed as low risk with a fraud risk score of 0.010, indicating that no dominant r...
  Escalation notice injected: False



c:\Users\sathi\fraud-platform\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


TxnID: 3518495      | Score: 0.0200 | Tier: LOW RISK       | Actual: LEGIT  | Latency: 14644ms
  Brief   : This transaction has been assessed as low risk with a fraud risk score of 0.020, indicating that no dominant r...
  Escalation notice injected: False


--- SUMMARY ---
Sample accuracy (20 cases)   : 65%
Avg latency (with narrative) : 14853ms
Max latency (with narrative) : 17576ms
Tier distribution            : {'LOW RISK': 17, 'MODERATE RISK': 3}


=== WITHOUT NARRATIVE (ML pipeline only) ===



c:\Users\sathi\fraud-platform\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\sathi\fraud-platform\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\sathi\fraud-platform\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\sathi\fraud-platform\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\sathi\fraud-platform\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestCla

Avg latency (ML only) : 13237ms
Max latency (ML only) : 14225ms
LLM overhead (avg)    : 1616ms


In [3]:
import joblib

model     = joblib.load('../models/fraud_model.pkl')
feat_cols = joblib.load('../models/feature_cols.pkl')

trained_cols = list(model.feature_names_in_)
print(f"trained: {len(trained_cols)}  |  feat_cols: {len(feat_cols)}")
print(f"Exact match: {trained_cols == feat_cols}")

if trained_cols != feat_cols:
    for i, (a, b) in enumerate(zip(trained_cols, feat_cols)):
        if a != b:
            print(f"First mismatch at index {i}: trained='{a}'  feat_cols='{b}'")
            break
    print("In feat_cols but not trained:", set(feat_cols) - set(trained_cols))
    print("In trained but not feat_cols:", set(trained_cols) - set(feat_cols))

print(feat_cols)
print(trained_cols)

trained: 105  |  feat_cols: 105
Exact match: True
['hour', 'day_of_week', 'log_amount', 'amount_to_card_mean', 'email_domain_mismatch', 'purchaser_email_risk', 'p_email_freq', 'is_mobile', 'has_identity', 'card2_freq', 'card3_freq', 'card4_freq', 'card5_freq', 'card6_freq', 'uid_count', 'uid_amt_mean', 'uid_amt_std', 'uid_amt_zscore', 'card1_amt_mean', 'amt_to_card1_mean', 'prod_C', 'prod_H', 'prod_R', 'prod_S', 'prod_W', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14', 'D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8', 'D9', 'D10', 'D11', 'D12', 'D13', 'D14', 'D15', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'dist1', 'dist2', 'addr1', 'addr2', 'id_01', 'id_02', 'id_03', 'id_04', 'id_05', 'id_06', 'id_07', 'id_08', 'id_09', 'id_10', 'id_11', 'id_13', 'id_14', 'id_17', 'id_18', 'id_19', 'id_20', 'id_21', 'id_22', 'id_24', 'id_25', 'id_26', 'id_32', 'id_12_freq', 'id_15_freq', 'id_16_freq', 'id_23_freq', 'id_27_freq', 'id_28_freq', 'id_29_f

In [14]:
import joblib
import pandas as pd

model     = joblib.load('../models/fraud_model.pkl')
feat_cols = joblib.load('../models/feature_cols.pkl')

df_test_eng = pd.read_csv('../data/processed/test_engineered.csv')

# the fraud cases your Day 20 run flagged LOW/MODERATE — pick a few
check_ids = [3518517, 3518582, 3518770, 3518797]

for tid in check_ids:
    row = df_test_eng[df_test_eng['TransactionID'] == tid]
    if row.empty:
        print(f"{tid}: NOT FOUND in test_engineered.csv — check split source")
        continue
    X_row = row[feat_cols].fillna(-999)
    prob  = model.predict_proba(X_row)[0, 1]
    print(f"{tid}: official engineered-row probability = {prob:.4f}")

3518517: official engineered-row probability = 0.0650
3518582: official engineered-row probability = 0.3750
3518770: official engineered-row probability = 0.0600
3518797: official engineered-row probability = 0.1400


In [10]:
df_raw_test = pd.read_csv('../data/processed/test.csv')
print(df_raw_test.shape, df_test_eng.shape)
print(set(df_raw_test['TransactionID']) == set(df_test_eng['TransactionID']))

(59054, 434) (59054, 475)
True


In [11]:
import joblib
import pandas as pd

model     = joblib.load('../models/fraud_model.pkl')
feat_cols = joblib.load('../models/feature_cols.pkl')
threshold = joblib.load('../models/threshold.pkl')

df_test_eng = pd.read_csv('../data/processed/test_engineered.csv')
X_test = df_test_eng[feat_cols].fillna(-999)
y_test = df_test_eng['isFraud']

probs = model.predict_proba(X_test)[:, 1]
preds = (probs >= threshold).astype(int)

from sklearn.metrics import classification_report, average_precision_score
print(classification_report(y_test, preds, target_names=['legit', 'fraud']))
print(f"PR-AUC: {average_precision_score(y_test, probs):.4f}")

              precision    recall  f1-score   support

       legit       0.98      0.99      0.98     56841
       fraud       0.60      0.50      0.54      2213

    accuracy                           0.97     59054
   macro avg       0.79      0.74      0.76     59054
weighted avg       0.97      0.97      0.97     59054

PR-AUC: 0.5330


In [16]:
from pipeline import predict_and_explain

for tid in check_ids:
    raw = df_raw_test[df_raw_test["TransactionID"] == tid].iloc[0].to_dict()

    pipe_score = predict_and_explain(
        raw,
        generate_narrative=False
    )["fraud_score"]

    row = df_test_eng[df_test_eng["TransactionID"] == tid]
    official_score = model.predict_proba(
        row[feat_cols].fillna(-999)
    )[0,1]

    print(
        tid,
        f"pipeline={pipe_score:.4f}",
        f"official={official_score:.4f}",
        f"diff={abs(pipe_score-official_score):.6f}"
    )

c:\Users\sathi\fraud-platform\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


3518517 pipeline=0.0650 official=0.0650 diff=0.000000


c:\Users\sathi\fraud-platform\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


3518582 pipeline=0.3900 official=0.3750 diff=0.015000


c:\Users\sathi\fraud-platform\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


3518770 pipeline=0.0650 official=0.0600 diff=0.005000


c:\Users\sathi\fraud-platform\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


3518797 pipeline=0.1300 official=0.1400 diff=0.010000
